Inspecting dataset images to better adapt for predictions

In [3]:
%pip install Pillow

Note: you may need to restart the kernel to use updated packages.


In [4]:
from PIL import Image
import os

In [5]:
def check_image_color_mode(img :Image):
    """Checks if an image is RGB or grayscale."""
    try:
        mode = img.mode
        if mode == 'RGB':
            return 'RGB'
        elif mode == 'L':  # 'L' mode represents grayscale
            return 'Grayscale'
        else:
            return f'Other ({mode})'  # Handle other color modes if necessary
    except Exception as e:
        return f"Error: {e}"

def check_dataset_images(dataset_folder):
    """Checks color modes of all images in a folder."""
    for filename in os.listdir(dataset_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            image_path = os.path.join(dataset_folder, filename)
            image = Image.open(image_path)
            color_mode = check_image_color_mode(image)
            print(f"{filename}: {color_mode}")

dataset_folder = 'BuilderFormer-4-4/valid/images'
check_dataset_images(dataset_folder)

APT_FP_SPA_034570688_PNG_jpg.rf.2559ee1af6f51c4a3dbebc34b0eb4a25.jpg: RGB
APT_FP_SPA_034570688_PNG_jpg.rf.afa45fcb3a51ee056da0a725c5244615.jpg: RGB
APT_FP_SPA_036906529_PNG_jpg.rf.f45e0252785ed0b8e7a4e1fa733759e4.jpg: RGB
APT_FP_SPA_037105612_PNG_jpg.rf.d9e9343ba7150b18c9a403629375d040.jpg: RGB
APT_FP_SPA_038082282_PNG_jpg.rf.f476cbe1ea2d894a22bdffbd717cb605.jpg: RGB
APT_FP_SPA_041684981_PNG_jpg.rf.aed1167966a26eccaaed22fb726ea28c.jpg: RGB
APT_FP_SPA_043277517_PNG_jpg.rf.4876e9e38ce3d8eccf396a745f710462.jpg: RGB
APT_FP_SPA_053461291_PNG_jpg.rf.c3db0fb52e47c9a7f5a0a2807ffbe8b7.jpg: RGB
APT_FP_SPA_054208404_PNG_jpg.rf.73ffecd4387ecb7a6faf9e7778993782.jpg: RGB
APT_FP_SPA_057345978_PNG_jpg.rf.67e9a04323e197ec88ceb1dc98adb9bb.jpg: RGB
APT_FP_SPA_057917679_PNG_jpg.rf.a449240d8dd8cecd5cab23af268ff932.jpg: RGB
APT_FP_SPA_060628957_PNG_jpg.rf.8546fe96edcf57d39b490f79d774c36a.jpg: RGB
APT_FP_SPA_060628957_PNG_jpg.rf.b0a90ded02600de9022f0a51986fc750.jpg: RGB
APT_FP_SPA_063117754_PNG_jpg.rf.61419f

Adjusting the test image

In [6]:
import cv2
import numpy as np

new_size = (640, 640) #based on the trainin data size

def pre_process(image : Image, size = (640, 640)):
    """Pre-processes the image to the same standards as the training data"""

    #converting to rgb if not rgb
    color_mode = check_image_color_mode(image)
    # print(color_mode)
    if color_mode != "RGB":
        image = image.convert("RGB") 

    #stripping auto-rotation
    data = list(image.getdata())
    image_stripped = Image.new(image.mode, image.size)
    image_stripped.putdata(data)

    # display(image_stripped)
    # print(image_stripped.size)

    #resizing
    old_size = image_stripped.size  # old_size[0] is in (width, height) format

    ratio = float(size[0])/max(old_size)
    new_size = tuple([int(x*ratio) for x in old_size])
    image_stripped.thumbnail(new_size)

    #im = im.resize(new_size, Image.ANTIALIAS)
    # create a new image and paste the resized on it

    new_im = Image.new("RGB", size, color="white")
    new_im.paste(image_stripped, ((size[0]-new_size[0])//2,
                        (size[1]-new_size[1])//2))

    # display(new_im)
    # print(new_im.size)

#adjust the size


In [7]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [8]:
from ultralytics import YOLO

image_loc = "my_custom/image (16).png"
img_test = Image.open(image_loc)

model_link = "C:/Users/Linos/Documents/Coding_practice/Dirbtinis Intelektas ir Python/Intelligent-Architecture/runs/segment/train7/weights/best.pt"
model = YOLO(model_link)

results = model.predict(img_test)


0: 448x640 1 background, 1 space_bedroom, 1 space_kitchen, 1 space_living_room, 7 space_others, 2 space_toilets, 68.8ms
Speed: 4.8ms preprocess, 68.8ms inference, 1217.3ms postprocess per image at shape (1, 3, 448, 640)


In [9]:
import numpy as np
from pathlib import Path
import cv2

In [16]:
result = results[0].cpu()
result.show()

In [20]:
# Iterate detection results 
img = np.copy(result.orig_img)

results_configured = []

# Iterate each object contour 
for i, contour in enumerate(result):
    label = contour.names[contour.boxes.cls.tolist().pop()]

    b_mask = np.zeros(img.shape[:2], np.uint8)

    # Create contour mask 
    contour = contour.masks.xy.pop().astype(np.int32).reshape(-1, 1, 2)
    _ = cv2.drawContours(b_mask, [contour], -1, 255, cv2.FILLED)

    print(label)

    mask_pil = Image.fromarray(b_mask)
    #display(mask_pil)

    binary_matrix = (b_mask > 0).astype(np.uint8)   

    results_configured.append(dict(label = label, mask = binary_matrix))

print(results_configured[0])

res = np.unique(results_configured[0]["mask"])
print("Unique values in matrix are : " + str(res))
print("Mask sums: ",np.sum(results_configured[0]["mask"]))


space_toilet
background
space_other
space_other
space_other
space_other
space_other
space_other
space_toilet
space_kitchen
space_living_room
space_bedroom
space_other
{'label': 'space_toilet', 'mask': array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint8)}
Unique values in matrix are : [0 1]
Mask sums:  15042


In [22]:
input("Let's pretend you're a user drawing a line on the plan to calculate the scale of the plan")
line_pixels = None
line_meters = None
while True:
    user_input = input("Choose how long the line would be in pixels: ").strip()
    try:
        user_input = float(user_input)
        if user_input <= 0:
            raise Exception(f"{user_input} isn't a valid length measurement")
        line_pixels = user_input
        break
    except Exception as ex:
        print(ex)
        print("Please enter the measurement as a number")

while True:
    user_input = input("Choose what the real workd scale of the line would be in meters: ").strip()
    try:
        user_input = float(user_input)
        if user_input <= 0:
            raise Exception(f"{user_input} isn't a valid length measurement")
        line_meters = user_input
        break
    except Exception as ex:
        print(ex)
        print("Please enter the measurement as a number")

print(f"The line drawn on the plan is {line_pixels} pixels long")
print(f"The line from the plan is {line_meters} meters long in real life")
    

The line drawn on the plan is 100.0 pixels long
The line from the plan is 2.0 meters long in real life


In [13]:
# 1px(one mask pixel which is square) = (line_length/user_input)^2
def calculate_sq_m(line_px, line_m, mask_matrix):
    sq_m = np.sum(mask_matrix) * ((line_m/line_px)**2)
    return sq_m

In [25]:
for result in results_configured:
    print("Room name: ", result["label"])
    area = round(calculate_sq_m(line_pixels, line_meters, result["mask"]), 3)
    result["area"] = area
    print(f"Room size: {result["area"] } m^2.")

Room name:  space_toilet
Room size: 6.017 m^2.
Room name:  background
Room size: 382.671 m^2.
Room name:  space_other
Room size: 22.467 m^2.
Room name:  space_other
Room size: 23.586 m^2.
Room name:  space_other
Room size: 22.8 m^2.
Room name:  space_other
Room size: 1.235 m^2.
Room name:  space_other
Room size: 39.831 m^2.
Room name:  space_other
Room size: 1.31 m^2.
Room name:  space_toilet
Room size: 16.173 m^2.
Room name:  space_kitchen
Room size: 15.926 m^2.
Room name:  space_living_room
Room size: 95.144 m^2.
Room name:  space_bedroom
Room size: 27.826 m^2.
Room name:  space_other
Room size: 28.977 m^2.
